In [1]:
#!git clone https://github.com/whyhardt/SPICE.git

In [2]:
# !pip install -e SPICE

In [1]:
import sys
import matplotlib.pyplot as plt

from spice import SpiceEstimator, csv_to_dataset, plot_session, split_data_along_sessiondim
from spice.precoded import workingmemory

sys.path.append("../..")
from weinhardt2025.benchmarking.benchmarking_gru import Model as GRU, training
from weinhardt2025.benchmarking.benchmarking_dezfouli2019 import GQLModel

# For custom RNN
import torch

## Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [2]:
# Load your data
path_data = '../data/dezfouli2019/dezfouli2019.csv'
dataset = csv_to_dataset(file = path_data)

# structure of dataset:
# dataset has two main attributes: xs -> inputs; ys -> targets (next action)
# shape: (n_participants*n_blocks*n_experiments, n_trials, n_timesteps=1, features)
# features are (n_actions * action, n_actions * reward, n_additional_inputs * additional_input, timestep, trial, block, experiment, participant)

# in order to set up the participant embedding we have to compute the number of unique participants in our data 
# to get the number of participants n_participants we do:
n_participants = len(dataset.xs[..., -1].unique())
n_actions = dataset.ys.shape[-1]

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")
print(f"Number of additional inputs: {dataset.xs.shape[-1]-2*n_actions-3}")

Shape of dataset: torch.Size([1212, 202, 1, 9])
Number of participants: 101
Number of actions in dataset: 2
Number of additional inputs: 2


In [3]:
test_sessions = (3, 6, 9)
dataset_train, dataset_test = split_data_along_sessiondim(dataset, test_sessions)

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [7]:
# fitting without SINDy coefficients

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=n_actions,
        n_participants=n_participants,
        
        epochs=1000,
        warmup_steps=200,
        sindy_weight=0,
        
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

  1%|▋                                                               | 11/1000 [00:10<16:23,  1.01it/s, L(Train)=0.4309846, L(Val,RNN)=0.4347826, Conv=9.13e-03]

Training interrupted. Continuing with further operations...

Stage 2: SINDy refit


 19%|█▊        | 187/1000 [00:13<01:00, 13.37it/s, loss=0.0040603, n_params=17.00+/-0.00]



Training interrupted. Continuing with further operations...

Training results:
	L(Train, RNN): 0.4239832
	L(Val, RNN):   0.4293299

RNN training finished.
Training took 26.56 seconds.


In [4]:
path_spice = '../params/dezfouli2019/spice_dezfouli2019.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=2,
        n_participants=n_participants,
        
        epochs=1000,
        warmup_steps=200,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        verbose=True,
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_train.xs, dataset_train.ys)

In [5]:
estimator.load_spice(path_spice)

In [12]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_chosen[t+1]     = 0.397 1 + 0.608 value_reward_chosen[t] + -0.894 reward[t] + 0.452 reward[t-1] + 0.393 reward[t-2] + 0.208 reward[t-3] 
value_reward_not_chosen[t+1] = -0.391 1 + 0.244 value_reward_not_chosen[t] + -0.026 reward[t-1] 
value_choice[t+1]            = 0.8 value_choice[t] + 0.443 choice[t-1] + -0.48 choice[t-2] + 0.178 choice[t-3] + 0.23 choice[t-1]^2 + -0.406 choice[t-2]^2 


## Benchmarking

### Generalized Q-Learning Model by Dezfouli (2019)

In [6]:
# 1. stick to low effort 
# 2. two sets of params for low and high effort

gql = GQLModel(
    n_participants=n_participants,
    batch_first=True,
    )

path_gql = '../params/dezfouli2019/baseline_dezfouli2019.pkl'

In [ ]:
epochs = 1000
optimizer = torch.optim.Adam(params=gql.parameters(), lr=0.01)

gql = training(
    model=gql,
    optimizer=optimizer,
    epochs=epochs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    device=torch.device('cpu'),
)

torch.save(gql.state_dict(), path_gql)

In [7]:
gql.load_state_dict(torch.load(path_gql, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [8]:
gru = GRU(n_actions)
path_gru = '../../weinhardt2025/params/dezfouli2019/gru_dezfouli2019.pkl'

In [9]:
epochs = 1000
optimizer = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 0.7302827835083008; L(Test): 0.6795499920845032
Epoch 2/1000: L(Train): 0.6816280484199524; L(Test): 0.6379129886627197
Epoch 3/1000: L(Train): 0.6388071179389954; L(Test): 0.5994971990585327
Epoch 4/1000: L(Train): 0.599480926990509; L(Test): 0.5627208948135376
Epoch 5/1000: L(Train): 0.5617916584014893; L(Test): 0.5274049043655396
Epoch 6/1000: L(Train): 0.5249030590057373; L(Test): 0.4951792359352112
Epoch 7/1000: L(Train): 0.4926966428756714; L(Test): 0.46951669454574585
Epoch 8/1000: L(Train): 0.46528157591819763; L(Test): 0.4543282389640808
Epoch 9/1000: L(Train): 0.44880005717277527; L(Test): 0.4504145681858063
Epoch 10/1000: L(Train): 0.443270742893219; L(Test): 0.45426174998283386
Epoch 11/1000: L(Train): 0.4454241693019867; L(Test): 0.46066585183143616
Epoch 12/1000: L(Train): 0.45136427879333496; L(Test): 0.4658443033695221
Epoch 13/1000: L(Train): 0.4546540379524231; L(Test): 0.468097060918808
Epoch 14/1000: L(Train): 0.45767149329185486; L(Test): 0.

In [9]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [6]:
from weinhardt2025.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2025.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2025.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals

## Analysis model evaluation

In [11]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=gql.to(torch.device('cpu')),
    gru_model=gru.to(torch.device('cpu')),
)

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,(std),AIC,BIC
Benchmark,0.717437,0.160890,12.000000,0.000000,0.332070,0.241141,0.881060,1.168477
GRU,0.753196,0.136911,1746.000000,0.000000,0.283430,0.192129,32.128613,73.947807
SPICE-RNN,0.745232,0.135882,50722.000000,0.000000,0.294060,0.189999,917.469849,2132.333984
SPICE,0.744315,0.136033,14.653465,1.934878,0.295291,0.190326,0.855467,1.207621


## Analysis coefficient distributions

In [12]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='../data/dezfouli2019/',
)

Extracting coefficient data...
  Ensemble=10, Participants=101, Experiments=1, Modules=3, Total terms=57

ANALYSIS 1: Ensemble Consistency
  Mean presence agreement: 1.000
  Mean presence rate:      0.257
  Mean CV:                 0.437
  Ensemble spread plots saved.
  Ensemble CV heatmaps saved.

ANALYSIS 2: Coefficient Distributions
  Terms with >50% presence: 16 / 57
  Terms with 0% presence:   31 / 57
  Violin plots saved.
  Presence rate bar chart saved.
  Experiment comparison skipped (X=1).
  Sparsity heatmaps saved.

All results saved to: ../data/dezfouli2019/


(                     module                                 term  term_index  \
 0       value_reward_chosen                                    1           0   
 1       value_reward_chosen                  value_reward_chosen           1   
 2       value_reward_chosen                            reward[t]           2   
 3       value_reward_chosen                          reward[t-1]           3   
 4       value_reward_chosen                          reward[t-2]           4   
 5       value_reward_chosen                          reward[t-3]           5   
 6       value_reward_chosen                value_reward_chosen^2           6   
 7       value_reward_chosen        value_reward_chosen*reward[t]           7   
 8       value_reward_chosen      value_reward_chosen*reward[t-1]           8   
 9       value_reward_chosen      value_reward_chosen*reward[t-2]           9   
 10      value_reward_chosen      value_reward_chosen*reward[t-3]          10   
 11      value_reward_chosen

## Analysis Individual Differences

In [13]:
analysis_coefficients_individuals(
    criterion="diag",
    analysis="disc",  # also: "cont"
    reference="Control",  # only necessary if analysis="disct"
    
    path_data=path_data,
    
    spice_model=estimator,
)

STEP 1: Preparing data
Extracted 57 SINDy coefficient columns for 101 participants.
After merge: 101 participants with criterion 'diag'.

STEP 2: Regression analysis

Discrete analysis: reference='Control', comparisons=['Bipolar', 'Depression']
  Bipolar: n=33
  Control: n=34
  Depression: n=34

Analysed 23 coefficients.

Control vs Bipolar: 7 significant coefficients
  Value Reward Chosen 1: OR=2.506, p=0.0061 ** (higher in Control)
  Value Reward Chosen Value Reward Chosen: OR=2.667, p=0.0010 *** (higher in Control)
  Value Reward Chosen Reward[T-3]: OR=2.667, p=0.0010 *** (higher in Control)
  Value Reward Chosen Value Reward Chosen^2: OR=2.622, p=0.0001 *** (higher in Control)
  Value Reward Chosen Reward[T]^2: OR=2.771, p=0.0002 *** (higher in Control)

Control vs Depression: 6 significant coefficients
  Value Reward Chosen Value Reward Chosen: OR=2.461, p=0.0028 ** (higher in Control)
  Value Reward Chosen Reward[T-3]: OR=2.302, p=0.0064 ** (higher in Control)
  Value Reward Chos